In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [3]:
data_train = pd.read_csv('/home/bigbang/machinelearning_algorithms/nn_train.csv')
X_train = data_train.iloc[:, 1:-2].values  
y_train = data_train.iloc[:, -1].values 
y_train = y_train.reshape(-1,1)

data_test = pd.read_csv('/home/bigbang/machinelearning_algorithms/nn_test.csv')
X_test = data_test.iloc[:, 1:].values 

# Normalize training data using Min-Max Normalization
def min_max_normalize(X):
    X_min = np.min(X, axis=0)
    X_max = np.max(X, axis=0)
    return (X - X_min) / (X_max - X_min)

# Normalize training data
X_train = min_max_normalize(X_train)
X_test = min_max_normalize(X_test)


print("Test data after Min-Max normalization:")
print(X_test[:1])  

print("Training data (first 5 rows):")
print(X_train[:1])
print(y_train[:1])

Test data after Min-Max normalization:
[[0.20392157 0.14117647 0.09019608 ... 0.31372549 0.1254902  0.11764706]]
Training data (first 5 rows):
[[0.31764706 0.42745098 0.35686275 ... 0.36078431 0.30588235 0.30588235]]
[[5]]


In [4]:
num_classes = 10
y_train_one_hot = np.zeros((y_train.size, num_classes))
y_train_one_hot[np.arange(y_train.size), y_train -1] = 1 


def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))  # Subtract max for numerical stability
    return expZ / np.sum(expZ, axis=1, keepdims=True)  # Use keepdims=True to maintain shape

def softmax_loss(Y_true, Y_pred):
    epsilon = 1e-8
    return -np.mean(np.sum(Y_true * np.log(Y_pred + epsilon)))

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def sigmoid_derivate(a):
    return a * (1-a) 

input_size = X_train.shape[1]
hidden_size = 128 
output_size = num_classes 
alpha = 0.01

np.random.seed(42)
W1 = np.random.randn(input_size , hidden_size)
b1 = np.zeros((1,hidden_size))
W2 = np.random.randn(hidden_size, output_size)
b2 = np.zeros((1,output_size))

epochs = 500

for i in range(epochs):
    Z1 = np.dot(X_train,W1) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    # Compute loss
    loss = softmax_loss(y_train_one_hot, A2)
    
    # Backpropagation
    dZ2 = A2 - y_train_one_hot
    dW2 = np.dot(A1.T, dZ2) / X_train.shape[0]
    db2 = np.sum(dZ2, axis=0) / X_train.shape[0]
    
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivate(A1)
    dW1 = np.dot(X_train.T, dZ1) / X_train.shape[0]
    db1 = np.sum(dZ1, axis=0) / X_train.shape[0]
    
    # Update weights and biases
    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2
    
    if i % 50 == 0:
        print(f"Epoch {i}, Loss: {loss}")


Epoch 0, Loss: 8250457.075006397
Epoch 50, Loss: 5050535.637975853
Epoch 100, Loss: 3213144.330388398
Epoch 150, Loss: 2410267.657104944
Epoch 200, Loss: 2164579.2681361246
Epoch 250, Loss: 2116732.3480816907
Epoch 300, Loss: 2116245.082624222
Epoch 350, Loss: 1983224.2320292168
Epoch 400, Loss: 1953216.311894636
Epoch 450, Loss: 1961344.9617259963


In [5]:
# Prediction on test data
Z1_test = np.dot(X_test, W1) + b1
A1_test = sigmoid(Z1_test)
Z2_test = np.dot(A1_test, W2) + b2
A2_test = softmax(Z2_test)
predictions = np.argmax(A2_test, axis=1) + 1  # Convert back to class labels 1 to 10

# Display results
print("Sample predictions:", predictions[:5])

Sample predictions: [5 3 1 5 6]


In [ ]:
def compute_f1_score(y_true, y_pred, num_classes=10):
    
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1_scores = np.zeros(num_classes)
    
    for class_label in range(1, num_classes + 1):
        true_positive = np.sum((y_pred == class_label) & (y_true == class_label))
        false_positive = np.sum((y_pred == class_label) & (y_true != class_label))
        false_negative = np.sum((y_pred != class_label) & (y_true == class_label))
        
        if true_positive + false_positive > 0:
            precision[class_label - 1] = true_positive / (true_positive + false_positive)
        if true_positive + false_negative > 0:
            recall[class_label - 1] = true_positive / (true_positive + false_negative)
        
        if precision[class_label - 1] + recall[class_label - 1] > 0:
            f1_scores[class_label - 1] = 2 * (precision[class_label - 1] * recall[class_label - 1]) / (precision[class_label - 1] + recall[class_label - 1])
    
    # Compute macro-averaged F1 score
    macro_f1 = np.mean(f1_scores)
    return f1_scores, macro_f1

# Prediction on training data
Z1_train = np.dot(X_train, W1) + b1
A1_train = sigmoid(Z1_train)
Z2_train = np.dot(A1_train, W2) + b2
A2_train = softmax(Z2_train)
train_predictions = np.argmax(A2_train, axis=1) + 1  # Convert back to class labels 1 to 10

# Compute F1 score for training data
f1_scores, macro_f1 = compute_f1_score(y_train, train_predictions, num_classes=num_classes)
print(f"F1 Scores for each class: {f1_scores}")
print(f"Macro-averaged F1 Score: {macro_f1}")